In [1]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import train_test_split

In [2]:
#read data
X_full = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/train.csv')
X_test_full = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/test.csv')

#enumerate the Heart Disease field. 1=Presence 0=Absence
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
X_full['Heart Disease'] = le.fit_transform(X_full['Heart Disease'])

#obtain target and predictors
y = X_full['Heart Disease']
features = ['id','Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression','Slope of ST','Number of vessels fluro','Thallium']
X = X_full[features].copy()
X_test = X_test_full[features].copy()

#break validation from training
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size = 0.8, test_size = 0.2, random_state=0)

In [3]:
#asses different regression models
from sklearn.ensemble import RandomForestClassifier

model_1 = RandomForestClassifier(n_estimators=50, random_state=0, n_jobs=-1)
model_2 = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
model_3 = RandomForestClassifier(n_estimators=100, criterion='log_loss', random_state=0, n_jobs=-1)
model_4 = RandomForestClassifier(n_estimators=200, min_samples_split=20, random_state=0, n_jobs=-1)
model_5 = RandomForestClassifier(n_estimators=100, max_depth=7, random_state=0, n_jobs=-1)

#prep for assessment
models = [model_1, model_2, model_3, model_4, model_5]

#Import metrics
from sklearn.metrics import accuracy_score

#function for comparing different models
def score_model(model, X_t=X_train, X_v=X_val, y_t=y_train, y_v=y_val):
    model.fit(X_t, y_t)
    preds = model.predict(X_v)
    return accuracy_score(y_v, preds)

#run assessment
for i in range(0, len(models)):
    mae = score_model(models[i])
    print(f"Model {i+1} Accuracy: {mae:.4f}")

Model 1 Accuracy: 0.8799
Model 2 Accuracy: 0.8810
Model 3 Accuracy: 0.8811
Model 4 Accuracy: 0.8835
Model 5 Accuracy: 0.8800


In [4]:
#model 4 most accurate
model_4.fit(X,y)

#prediction test
preds_test = model_4.predict(X_test)

# Save predictions in format used for competition scoring
output = pd.DataFrame({'id': X_test['id'],
                       'Heart Disease': preds_test})


output.to_csv('submission.csv', index=False)